# Task 3 — Gender G-D1 mild darkening screen

Run All performs the GPU prerequisites first, then only screen folds **0 and 4** if those checks pass. It trains a new scratch model. G2 stays rejected and is a comparison model only. No confirmation folds or final test rows are trained here.

Use the same GPU/build as the saved G2 runs (currently NVIDIA L4). The prerequisite checks this automatically. Upload or publish this code to the selected repository branch before using the Colab setup below. No local PyTorch install is required.


## 1. Colab GPU and repository


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Commit: 05a523f257e15575859f57608d08c4ff351185d5


## 2. Official teacher data and canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. Frozen candidate

Keep G2's exact ±2-pixel translation. After translation, darken the entire image with probability 0.25, using brightness Uniform(0.90, 1.00). The darkening RNG is separate from translation and sampling. Keep GeM p=3, batch 128, FP32, original fold-training RGB statistics, cross-entropy, AdamW, schedule, seed 2753 and 30 epochs. The model starts from scratch.

Use normal GPU execution with FP32 and batch 128. GPU allocation must stay **below 3,000,000,000 bytes (3 GB)**. Forward/backward time, training time and latency are recorded without speed caps. A zero-step GPU probe includes reserve memory for AdamW's two moments and a temporary. Actual training memory must also pass the same limit.

The original CPU-offload plan stopped before training: matching outputs passed and memory was 273,277,440 bytes, but the probe was 7.595× slower than normal GPU execution. This revised plan removes offload and speed caps at the user's request. Its artifacts use a separate `gpu_v2` directory, preserving the old failed result.

The diagnostics reproduce all ten saved E6/G2 clean OOF bundles first, then measure brightness 0.85/0.90/0.95/1.00 and all 25 small shifts. Save paired probabilities, per-class scores and prediction flips. These are development diagnostics. The held-out test stays sealed.


In [3]:
from fashion.train.task3_gender_repair_preflight import prepare_gender_repair
from fashion.train.task3_gender_repair import run_gender_repair

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
PREREQUISITE_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_translation_mild_darkening_gpu_v2/prerequisites"
PREREQUISITE_PATH = PREREQUISITE_DIR / "prerequisites.json"
for path in (G2_DIR, E6_DIR, DRIVE_REGISTRY):
    if not path.exists():
        raise FileNotFoundError(f"Complete saved source evidence is required: {path}")

prerequisites = prepare_gender_repair(
    g2_directory=G2_DIR, e6_directory=E6_DIR, registry_path=DRIVE_REGISTRY,
    report_directory=PREREQUISITE_DIR, root=REPO_DIR, device_name="cuda",
)
print("Ready:", prerequisites["ready"])
print("Memory probe:", prerequisites["memory_profile"])


[gender-repair] E6 fold 0 clean: 0.719640
[gender-repair] E6 fold 0 brightness_0.85: 0.548527
[gender-repair] E6 fold 0 brightness_0.90: 0.610958
[gender-repair] E6 fold 0 brightness_0.95: 0.652671
[gender-repair] E6 fold 0 brightness_1.00: 0.719640
[gender-repair] E6 fold 0 shift_-2_-2: 0.603010
[gender-repair] E6 fold 0 shift_-2_-1: 0.662790
[gender-repair] E6 fold 0 shift_-2_0: 0.670700
[gender-repair] E6 fold 0 shift_-2_1: 0.641712
[gender-repair] E6 fold 0 shift_-2_2: 0.580416
[gender-repair] E6 fold 0 shift_-1_-2: 0.630788
[gender-repair] E6 fold 0 shift_-1_-1: 0.675539
[gender-repair] E6 fold 0 shift_-1_0: 0.708347
[gender-repair] E6 fold 0 shift_-1_1: 0.678111
[gender-repair] E6 fold 0 shift_-1_2: 0.621738
[gender-repair] E6 fold 0 shift_0_-2: 0.642526
[gender-repair] E6 fold 0 shift_0_-1: 0.690080
[gender-repair] E6 fold 0 shift_0_0: 0.719640
[gender-repair] E6 fold 0 shift_0_1: 0.699218
[gender-repair] E6 fold 0 shift_0_2: 0.636303
[gender-repair] E6 fold 0 shift_1_-2: 0.6194

## 4. Screen folds 0 and 4, then stop

Compared with G2: pooled clean F1 loss at most 0.005; each fold loss at most 0.010; each pooled class loss at most 0.020; paired family CI lower bound at least −0.005. Dark-induced loss must improve at least 0.030 on average and improve in both folds. Translation loss may worsen at most 0.010.

Every fold must keep 390,181 parameters, peak GPU allocation below 3,000,000,000 bytes. Time and latency are reported without caps. Registry and source integrity are checked. Existing matching runs are reused only after verification. Bootstrap uses 10,000 whole-family paired draws within folds, seed 2753.


In [4]:
gender_gd1 = run_gender_repair(
    g2_directory=G2_DIR, e6_directory=E6_DIR, source_registry_path=DRIVE_REGISTRY,
    output_root=DRIVE_TASK_DIR, registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,), prerequisite_path=PREREQUISITE_PATH,
    root=REPO_DIR, phase="screen", device_name="cuda",
)
print("Screen decision:", gender_gd1["status"])
for check in gender_gd1["checks"]:
    if check["status"] != "pass":
        print(check)


[task3] preparing target=gender fold=0: train=26,220 (before selection=26,220), validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_translation_mild_darkening_gender_smallcnngem3_f0_s2753_a4cc8d75f3fb_20260905T051933Z2d5d0f; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.5858 train_macro_f1=0.4510 validation_loss=0.5721 validation_macro_f1=0.3469
[task3] target=gender fold=0 epoch=2/30 train_loss=0.4459 train_macro_f1=0.5989 validation_loss=0.4354 validation_macro_f1=0.6640
[task3] target=gender fold=0 epoch=3/30 train_loss=0.3921 train_macro_f1=0.6563 validation_loss=0.3846 validation_macro_f1=0.6745
[task3] target=gender fold=0 epoch=4/30 train_loss=0.3579 train_macro_f1=0.6857 validation_loss=0.3959 validation_macro_f1=0.5549
[task3] target=gender fold=0 epoch=5/30 train_loss=0.3312 train_macro_f1=0.7172 validation_loss=0.39

## 5. Confirmation is a separate later decision

Stop after the screen. If it passes and confirmation is requested, use the same `run_gender_repair` arguments with `phase="confirmation"`. That phase verifies the passing screen again, reuses folds 0/4 and trains only fresh folds 1/2/3. It cannot proceed from an absent or failed screen.

The fresh folds must improve pooled clean F1 over E6, have a positive paired 95% lower bound, improve at least two folds and lose no more than 0.005 on any fold. All five must retain every original G2 rule, plus the pooled screen margins against G2 and the revised memory limit (speed has no cap). A pass opens the planned fixed-seed confirmation; it does not make reused development folds independent test evidence.
